# Social Capital — The Best-Measured Quantity Here, and It Still Cannot Be Ranked

Handbook §7.2 makes a claim that has been tested twice and could plausibly have been an
artifact both times: **rank failure at fine geography is structural — a consequence of
multiplicity — rather than a precision problem.** D04 tested it on a *modelled* estimate and
D10 on a *measured* one, and each time a reader could object that the precision simply was
not good enough yet.

This notebook removes that objection. Economic connectedness at county level is, by a wide
margin, the most precisely measured quantity in this corpus: its confidence intervals are
about four per cent of the between-county spread, roughly twenty times better than the best
case in D10. And there are twenty-three times as many units.

If precision were the binding constraint, this is where ranking would start working.

Along the way the source introduces a regime of its own. It is the first here protected by
**formal differential privacy with a published budget** — Laplace noise at ε = 8 — and the
first whose published standard error is documented as containing statistical error and
disclosure error *added together in one column*. Part of the uncertainty in this dataset is
therefore not a fact about the world but a **policy choice**, and §2 shows that the two parts
cannot be pulled apart from what was released.

## Learning Objectives

1. **Distinguish two kinds of absence** — a unit removed before publication, and a unit
   present with an empty estimate — and know why a file can only ever show you one of them.
2. **Read a platform universe** as something other than a sample of a population.
3. **Recognise uncertainty that is partly a policy parameter**, and test whether its
   components can be separated from what the publisher released.
4. **Settle §7.2** by testing it where precision is not the constraint.
5. **Say what high precision does buy**, which turns out to be the extremes rather than the
   order.

## Prerequisites

- Handbook §7.1 and §7.2. This notebook is that argument's decisive test.
- D04 (modelled estimate, reliability 0.853, zero determined ranks) and D10 (measured
  estimate, reliability 0.916, interval narrower than the spread, zero determined ranks).
  Together with this one they form the three-point series in §4 below.
- Handbook §1.9 for the Opportunity Atlas, whose noise is acknowledged but unquantified.
  Here it is quantified, which is what makes §2 possible.

## Data Sources

| file | unit | published universe |
|---|---|---|
| `social_capital_county.csv` | county | 3,096 |
| `social_capital_zip.csv` | ZIP code | 29,062 |
| `social_capital_college.csv` | college | 2,673 |
| `social_capital_high_school.csv` | high school | 21,483 |

From the Social Capital Atlas (Meta Platforms and Opportunity Insights), the data behind
Chetty et al., *Nature*, 2022. Universe: **Facebook users aged 25–44, active in the previous
30 days, with at least 100 US friends and a valid residential ZIP as of 28 May 2022.**

Two access notes. The HDX dataset landing page answers a plain request with **HTTP 202 and a
zero-length body** — the firewall signature catalogued in handbook §2 — while the CKAN API and
the resource download paths are not challenged, so the resource ids are pinned in the client
rather than scraped. And unlike the Opportunity Atlas files in D10, **redistribution here is
permitted**: Meta's licence allows use and redistribution with or without modification,
provided the licence travels with any copy and the data are not combined with other sources in
a way that could identify an individual. The runtime fetch is kept anyway, so that a vintage
and a payload hash are recorded on every use.

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.reliability import rank_intervals
from krl_tutorials_data.sources import (
    SOCIAL_CAPITAL_PRIVACY,
    SOCIAL_CAPITAL_UNIVERSE,
    SOCIAL_CAPITAL_UNIVERSE_NOTE,
    SOCIAL_CAPITAL_VINTAGE,
    fetch_social_capital_atlas,
)
from krl_tutorials_data.uncertainty import Estimate

#: Per geography: the size proxy available in the file, the estimate, and its
#: standard error. None of the size columns is the count the noise formula uses;
#: that count is never published, which is the subject of section 2.
SPEC = {
    "county":      ("num_below_p50", "ec_county", "ec_se_county"),
    "zip":         ("num_below_p50", "ec_zip", "ec_se_zip"),
    "college":     ("mean_students_per_cohort", "ec_own_ses_college", "ec_own_ses_se_college"),
    "high_school": ("students_9_to_12", "ec_own_ses_hs", "ec_own_ses_se_hs"),
}

CONFIDENCE_Z = 1.96          # 95%, the convention these standard errors invite
RELIABILITY_FLOOR = 0.7      # the provider-profiling publishing convention, handbook 7.1

pd.set_option("display.width", 112)

print(f"vintage  : {SOCIAL_CAPITAL_VINTAGE}")
print(f"universe : {SOCIAL_CAPITAL_UNIVERSE_NOTE}")
print()
print(f"privacy  : {SOCIAL_CAPITAL_PRIVACY}")

vintage  : 2022-07-31 release; network snapshot 2022-05-28
universe : Facebook users aged 25-44, active in the previous 30 days, with at least 100 US friends and a valid residential ZIP as of 2022-05-28. This is a platform universe, not a demographic one: coverage varies across places for reasons plausibly correlated with the social connectedness being measured, and no user count is published at any geography

privacy  : formal differential privacy: Laplace noise at epsilon = 8 with a smoothed sensitivity term, and minimum cell sizes of 100 low-SES and 100 high-SES users. The published standard-error columns are documented as 'incorporating both sampling error and error from the addition of noise to protect privacy' -- the two are added together and cannot be separated from the release, because the sensitivity term depends on the within-unit degree distribution, which is not published


In [2]:
"""Fetch all four geographies."""
frames = {}
provenance = {}
for geography in SPEC:
    frames[geography], provenance[geography] = fetch_social_capital_atlas(geography=geography)
    print(f"  {geography:12s} {len(frames[geography]):>6,} rows x "
          f"{len(frames[geography].columns):>2} columns")

print()
for note in provenance["zip"].notes:
    print(f"  - {note}")

  county        3,089 rows x 26 columns


  zip          23,028 rows x 23 columns


  college       2,586 rows x 22 columns


  high_school  17,525 rows x 21 columns

  - Social Capital Atlas, zip. Vintage: 2022-07-31 release; network snapshot 2022-05-28. There is one release and no vintage parameter.
  - Universe: Facebook users aged 25-44, active in the previous 30 days, with at least 100 US friends and a valid residential ZIP as of 2022-05-28. This is a platform universe, not a demographic one: coverage varies across places for reasons plausibly correlated with the social connectedness being measured, and no user count is published at any geography.
  - Disclosure protection: formal differential privacy: Laplace noise at epsilon = 8 with a smoothed sensitivity term, and minimum cell sizes of 100 low-SES and 100 high-SES users. The published standard-error columns are documented as 'incorporating both sampling error and error from the addition of noise to protect privacy' -- the two are added together and cannot be separated from the release, because the sensitivity term depends on the within-unit degree di

## 1. Two kinds of absence, and only one of them is visible

The minimum-cell-size rule — at least 100 below-median-SES and 100 above-median-SES users —
operates twice. It removes whole units from the release, and it blanks individual statistics
for units that survive. The second kind is visible in the file as an empty cell. **The first
kind leaves no trace at all**, which is why the published universe counts have to be carried
in the client from the release README rather than derived from the data.

The interesting part is not that suppression happens. It is that the rate varies by a factor
of roughly a hundred across four geographies in the same release, entirely as a function of
how small the units are.

In [3]:
"""How much was removed, and how much was blanked."""
rows = []
for geography, (size_col, ec_col, se_col) in SPEC.items():
    frame = frames[geography]
    universe = SOCIAL_CAPITAL_UNIVERSE[geography]
    published = len(frame)
    with_estimate = int(frame[ec_col].notna().sum())
    rows.append({
        "geography": geography,
        "universe": universe,
        "rows published": published,
        "removed before publication": universe - published,
        "removed %": 100 * (universe - published) / universe,
        "blank estimate in file": published - with_estimate,
        "blank %": 100 * (published - with_estimate) / published,
        "usable": with_estimate,
        "usable % of universe": 100 * with_estimate / universe,
    })

coverage = pd.DataFrame(rows).set_index("geography")
print(coverage.to_string(formatters={
    "universe": "{:,.0f}".format, "rows published": "{:,.0f}".format,
    "removed before publication": "{:,.0f}".format, "removed %": "{:.1f}".format,
    "blank estimate in file": "{:,.0f}".format, "blank %": "{:.1f}".format,
    "usable": "{:,.0f}".format, "usable % of universe": "{:.1f}".format}))

worst = coverage["removed %"].idxmax()
best = coverage["removed %"].idxmin()
print()
print(f"removal rate spans {coverage.loc[best, 'removed %']:.1f}% ({best}) to "
      f"{coverage.loc[worst, 'removed %']:.1f}% ({worst}), a factor of "
      f"{coverage.loc[worst, 'removed %'] / coverage.loc[best, 'removed %']:.0f}")

# The size columns are NOT the same quantity, and comparing them as served makes
# colleges look anomalous. `mean_students_per_cohort` counts one year; every other
# column counts the whole unit. Put them on one basis before reading the gradient.
COHORTS_PER_COLLEGE = 4
print()
print("median unit size, as served and on a comparable whole-unit basis:")
comparable = {}
for geography, (size_col, ec_col, se_col) in SPEC.items():
    served = float(frames[geography][size_col].dropna().median())
    adjusted = served * COHORTS_PER_COLLEGE if geography == "college" else served
    comparable[geography] = adjusted
    note = f"x{COHORTS_PER_COLLEGE} cohorts" if geography == "college" else ""
    print(f"  {geography:12s} {size_col:26s} served {served:>9,.0f}   "
          f"comparable {adjusted:>9,.0f}  {note}")

order_by_size = sorted(comparable, key=lambda g: -comparable[g])
order_by_removal = list(coverage["removed %"].sort_values().index)
print()
print(f"  largest unit first        : {order_by_size}")
print(f"  lowest removal rate first : {order_by_removal}")
print(f"  the two orderings agree   : {order_by_size == order_by_removal}")

            universe rows published removed before publication removed % blank estimate in file blank % usable usable % of universe
geography                                                                                                                          
county         3,096          3,089                          7       0.2                     71     2.3  3,018                 97.5
zip           29,062         23,028                      6,034      20.8                  4,048    17.6 18,980                 65.3
college        2,673          2,586                         87       3.3                    224     8.7  2,362                 88.4
high_school   21,483         17,525                      3,958      18.4                  3,448    19.7 14,077                 65.5

removal rate spans 0.2% (county) to 20.8% (zip), a factor of 92

median unit size, as served and on a comparable whole-unit basis:
  county       num_below_p50              served     3,353   comparable     

Read the removal column against the size column and one geography looks wrong: colleges lose
only a few per cent while ZIP codes lose a fifth, on apparently similar unit sizes. The
anomaly is at least partly in the columns rather than in the rule. **`mean_students_per_cohort`
counts one year of students; every other size column counts the whole unit.** Put on a
comparable basis, colleges are roughly four times larger than the raw figure suggests, which
moves them from looking anomalous to sitting where the rule would put them.

It does not fix everything, and the cell says so rather than rounding it off: ZIP codes and
high schools still swap places, with the slightly larger units losing slightly more. On a
two-point difference in removal rate between two geographies whose size proxies measure
different things and differ by under a fifth, that residual is not worth a story.

The part worth keeping is the cheaper lesson, and the one more likely to bite: four columns,
four different underlying quantities, no units recorded anywhere in the file, and a comparison
across them that looks perfectly reasonable and is not.

The substantive lesson is the one handbook §1.9 named for the Opportunity Atlas: **a uniform
threshold applied to unequal populations is a patterned deletion.** Small places and small
schools are removed; small places and small schools are not a random sample of places and
schools. Any statement about "high schools in America" built from this file is a statement
about the larger ones, and the file gives no way to characterise the fifth that is gone.

## 2. An uncertainty that is partly a policy choice

Now the part that makes this source its own regime. The release documents its disclosure
mechanism completely — Laplace noise, ε = 8, a smoothed sensitivity term, minimum cell sizes —
and then defines the standard-error columns as:

> "The standard error of economic connectedness, incorporating both sampling error and error
> from the addition of noise to protect privacy."

Everywhere else in this corpus, a published uncertainty is a statistical property of a sample:
it describes how much the world was not observed. Here one component of it describes how much
the publisher **chose** not to reveal. Epsilon is a dial. At ε = 1 the same underlying network
would have produced visibly larger standard errors; at ε = ∞, smaller ones. The number in the
column is part measurement and part governance.

That would be unremarkable if the two could be separated. The obvious test is how the standard
error scales with unit size: sampling error falls as `N^-0.5`, while Laplace noise whose scale
goes as `1/N` falls as `N^-1.0`. Fit one against the other and the mixture is revealed.

In [4]:
"""Try to separate the two components by how the standard error scales."""
scaling = []
for geography, (size_col, ec_col, se_col) in SPEC.items():
    frame = frames[geography][[size_col, ec_col, se_col]].dropna()
    frame = frame[(frame[size_col] > 0) & (frame[se_col] > 0)]
    fit = stats.linregress(
        np.log(frame[size_col].astype(float)), np.log(frame[se_col].astype(float))
    )
    scaling.append({
        "geography": geography, "n": len(frame), "slope": fit.slope,
        "slope se": fit.stderr, "r squared": fit.rvalue ** 2,
        "ci low": fit.slope - CONFIDENCE_Z * fit.stderr,
        "ci high": fit.slope + CONFIDENCE_Z * fit.stderr,
    })

scaling = pd.DataFrame(scaling).set_index("geography")
print("log(standard error) regressed on log(unit size proxy)")
print("  a pure sampling error would give -0.5; Laplace noise scaling as 1/N would give -1.0")
print()
print(scaling.to_string(formatters={
    "n": "{:,.0f}".format, "slope": "{:+.3f}".format, "slope se": "{:.4f}".format,
    "r squared": "{:.3f}".format, "ci low": "{:+.3f}".format, "ci high": "{:+.3f}".format}))

print()
print("every slope is shallower than -0.5, so neither hypothesis is supported as stated.")
print("before concluding anything about the mechanism, check the regressor.")
county = frames["county"][["num_below_p50", "pop2018", "ec_se_county"]].dropna()
county = county[(county["num_below_p50"] > 0) & (county["ec_se_county"] > 0)]
for proxy in ("num_below_p50", "pop2018"):
    fit = stats.linregress(
        np.log(county[proxy].astype(float)), np.log(county["ec_se_county"].astype(float))
    )
    print(f"  counties, log SE on log({proxy:14s}) slope={fit.slope:+.3f}  "
          f"R2={fit.rvalue ** 2:.3f}")
agreement = stats.pearsonr(np.log(county["num_below_p50"]), np.log(county["pop2018"]))
print(f"  the two proxies correlate r={agreement[0]:.3f} in logs, and give different slopes")

log(standard error) regressed on log(unit size proxy)
  a pure sampling error would give -0.5; Laplace noise scaling as 1/N would give -1.0

                 n  slope slope se r squared ci low ci high
geography                                                  
county       3,016 -0.308   0.0024     0.846 -0.313  -0.303
zip         18,969 -0.292   0.0014     0.687 -0.295  -0.289
college      2,065 -0.155   0.0058     0.260 -0.166  -0.144
high_school 14,077 -0.342   0.0016     0.759 -0.345  -0.339

every slope is shallower than -0.5, so neither hypothesis is supported as stated.
before concluding anything about the mechanism, check the regressor.
  counties, log SE on log(num_below_p50 ) slope=-0.300  R2=0.838
  counties, log SE on log(pop2018       ) slope=-0.264  R2=0.831
  the two proxies correlate r=0.961 in logs, and give different slopes


The test does not work, and the reason is worth more than the result would have been.

**Neither size column is the quantity the noise formula uses.** The Laplace scale depends on a
smoothed sensitivity term built from the *degree distribution of below-median-SES Facebook
users within each unit*. No user count is published at any geography, let alone a degree
distribution — and publishing one would leak exactly the network structure the noise exists to
protect. What the files carry instead are two external proxies: `num_below_p50` is children
with below-median parental income, taken from the Opportunity Atlas, and `pop2018` is ACS
population. They correlate at 0.961 in logs and still give
slopes 0.036 apart — a gap many times either standard error, which is what a pair of
imperfect stand-ins for an unobserved regressor does. Both are nowhere near either hypothesis.

**So the decomposition is not identifiable from the release**, and it is not identifiable by
design rather than by oversight. The honest statement is narrow and worth making precisely:

> The publisher documents the mechanism fully, states the budget, and combines the two error
> components into one column. An analyst can therefore know that part of the published
> uncertainty is a policy parameter, and cannot find out how much.

**[rule]** When a source combines statistical and disclosure error in one published figure,
record that it did. Do not estimate the split from published size variables unless the
publisher tells you the noise is a function of one of them — the tempting regression above is
exactly the kind of plausible, unfounded number this corpus exists to refuse.

## 3. The decisive test

Everything so far has been about the uncertainty. Now use it.

Handbook §7.2 argues that rank failure at fine geography is a multiplicity problem rather than
a precision problem. Two notebooks have supported that, and both had an obvious objection
available: their measurements were not very precise. This one is.

In [5]:
"""Reliability and rank identifiability on county economic connectedness."""
size_col, ec_col, se_col = SPEC["county"]
county_ec = frames["county"][["county_name", ec_col, se_col]].dropna().reset_index(drop=True)
value = county_ec[ec_col].to_numpy(float)
error = county_ec[se_col].to_numpy(float)
n_units = len(county_ec)

# DerSimonian-Laird moment estimator for the between-unit variance.
weight = 1.0 / error ** 2
pooled = (weight * value).sum() / weight.sum()
q_statistic = (weight * (value - pooled) ** 2).sum()
scale = weight.sum() - (weight ** 2).sum() / weight.sum()
tau_squared = max(0.0, (q_statistic - (n_units - 1)) / scale)
reliability = tau_squared / (tau_squared + error ** 2)

spread = value.max() - value.min()
half_width = CONFIDENCE_Z * np.median(error)

print(f"counties with a published estimate and standard error : {n_units:,}")
print(f"between-county spread                                 : {spread:.3f} "
      f"({value.min():.3f} to {value.max():.3f})")
print(f"median standard error                                 : {np.median(error):.4f}")
print(f"median 95% interval width                             : {2 * half_width:.4f}")
print(f"interval width as a fraction of the spread            : {2 * half_width / spread:.4f}")
print()
print(f"between-unit standard deviation tau                   : {np.sqrt(tau_squared):.4f}")
print(f"reliability lambda, median                            : {np.median(reliability):.4f}")
print(f"reliability lambda, minimum                            : {reliability.min():.4f}")
print(f"share clearing the {RELIABILITY_FLOOR} publishing convention  : "
      f"{(reliability >= RELIABILITY_FLOOR).mean():.1%}")

counties with a published estimate and standard error : 3,018
between-county spread                                 : 1.065 (0.295 to 1.360)
median standard error                                 : 0.0119
median 95% interval width                             : 0.0466
interval width as a fraction of the spread            : 0.0438

between-unit standard deviation tau                   : 0.1683
reliability lambda, median                            : 0.9950
reliability lambda, minimum                            : 0.9182
share clearing the 0.7 publishing convention  : 100.0%


In [6]:
"""How many of those ranks are actually determined?"""
estimates = {
    county_ec["county_name"][i]: Estimate(value=float(value[i]),
                                          moe=float(CONFIDENCE_Z * error[i]))
    for i in range(n_units)
}
intervals = rank_intervals(estimates)
span = np.array([r.worst - r.best + 1 for r in intervals])

print(f"units ranked                       : {n_units:,}")
print(f"ranks DETERMINED (span of exactly 1): {int((span == 1).sum())}")
print(f"median rank span                   : {np.median(span):,.0f} "
      f"({np.median(span) / n_units:.1%} of the table)")
print(f"tightest span achieved             : {span.min():,}")
print(f"units with a span under 10         : {int((span < 10).sum())} "
      f"({(span < 10).mean():.2%})")
print(f"units with a span under 100        : {int((span < 100).sum())} "
      f"({(span < 100).mean():.2%})")
print()
ordered = sorted(intervals, key=lambda r: r.point)
print("the five highest by point estimate, with the ranks their intervals permit:")
for r in ordered[:5]:
    print(f"  point {r.point:>4}   attainable [{r.best:>4} .. {r.worst:>4}]   "
          f"span {r.worst - r.best + 1:>4}   {r.label}")
print("the three lowest:")
for r in ordered[-3:]:
    print(f"  point {r.point:>4}   attainable [{r.best:>4} .. {r.worst:>4}]   "
          f"span {r.worst - r.best + 1:>4}   {r.label}")

units ranked                       : 3,018
ranks DETERMINED (span of exactly 1): 0
median rank span                   : 500 (16.6% of the table)
tightest span achieved             : 2
units with a span under 10         : 8 (0.27%)
units with a span under 100        : 80 (2.65%)

the five highest by point estimate, with the ranks their intervals permit:
  point    1   attainable [   1 ..    2]   span    2   Rockingham, New Hampshire
  point    2   attainable [   1 ..    3]   span    3   Douglas, Colorado
  point    3   attainable [   3 ..    7]   span    5   San Francisco, California
  point    4   attainable [   2 ..   26]   span   25   Morgan, Utah
  point    5   attainable [   3 ..   16]   span   14   Madison, Idaho
the three lowest:
  point 3016   attainable [3011 .. 3017]   span    7   Terrell, Georgia
  point 3017   attainable [3013 .. 3018]   span    6   Shannon, South Dakota
  point 3018   attainable [3017 .. 3018]   span    2   Todd, South Dakota


**Zero.** Out of three thousand counties, with confidence intervals about four per cent of the
spread they sit in and a reliability that clears the publishing convention everywhere, not one
county's rank is pinned down. The median county could occupy any of roughly five hundred
positions.

That settles §7.2 as firmly as this corpus can settle anything. The objection available to D04
and D10 — that precision was the binding constraint — does not survive a case with twenty
times better relative precision and twenty-three times more units. **Adding units makes
ranking worse, and adding precision does not rescue it**, because the number of comparisons a
rank has to win grows with the table while the evidence per comparison does not.

But the output above also contains a refinement that the earlier notebooks were too imprecise
to show, and it should not be lost in the headline.

**Precision buys the extremes.** A handful of counties have spans in the low single digits.
The top of the table and the bottom of the table *are* resolved: it is possible to say which
counties are among the very highest and which among the very lowest, and to be right. What is
not resolvable is everything in between — and "everything in between" is where nearly all of
the table lives.

This is the useful form of the §7.2 result, and it is the form to hand a policymaker: **you
may name the extremes; you may not publish the list.**

In [7]:
"""What the data do support: separation from a precision-weighted benchmark."""
z_score = (value - pooled) / error
print(f"precision-weighted national benchmark : {pooled:.4f}")
print(f"(economic connectedness is a ratio, so {1.0:.1f} would be parity between "
      f"below- and above-median-SES friendships)")
print()
for label, critical in (("95%", 1.96), ("99.8%", 3.09)):
    separated = int((np.abs(z_score) > critical).sum())
    print(f"  separated from the benchmark at {label:>6}: {separated:>5,} of {n_units:,} "
          f"({separated / n_units:.1%})   above {int((z_score > critical).sum()):>5,}   "
          f"below {int((z_score < -critical).sum()):>5,}")

print()
print("the same question asked of every geography, on its own units:")
for geography, (size_col, ec_col, se_col) in SPEC.items():
    unit = frames[geography][[ec_col, se_col]].dropna()
    if unit.empty:
        continue
    v = unit[ec_col].to_numpy(float); e = unit[se_col].to_numpy(float)
    w = 1.0 / e ** 2
    centre = (w * v).sum() / w.sum()
    ratio = 2 * CONFIDENCE_Z * np.median(e) / (v.max() - v.min())
    print(f"  {geography:12s} n={len(unit):>6,}  median EC={np.median(v):.3f}  "
          f"interval/spread={ratio:.3f}  separated at 95%: "
          f"{(np.abs((v - centre) / e) > 1.96).mean():.1%}")

precision-weighted national benchmark : 0.7844
(economic connectedness is a ratio, so 1.0 would be parity between below- and above-median-SES friendships)

  separated from the benchmark at    95%: 2,678 of 3,018 (88.7%)   above 1,487   below 1,191
  separated from the benchmark at  99.8%: 2,478 of 3,018 (82.1%)   above 1,376   below 1,102

the same question asked of every geography, on its own units:
  county       n= 3,018  median EC=0.807  interval/spread=0.044  separated at 95%: 88.7%
  zip          n=18,980  median EC=0.876  interval/spread=0.052  separated at 95%: 86.3%
  college      n= 2,362  median EC=1.446  interval/spread=0.074  separated at 95%: 86.8%
  high_school  n=14,077  median EC=1.057  interval/spread=0.087  separated at 95%: 79.9%


In [8]:
"""FINDINGS"""
findings = []

findings.append((
    "F1",
    f"The minimum-cell-size rule removes units before publication and the rate spans a factor "
    f"of {coverage.loc[worst, 'removed %'] / coverage.loc[best, 'removed %']:.0f} across four "
    f"geographies in one release: {coverage.loc['county', 'removed %']:.1f}% of counties, "
    f"{coverage.loc['college', 'removed %']:.1f}% of colleges, "
    f"{coverage.loc['high_school', 'removed %']:.1f}% of high schools and "
    f"{coverage.loc['zip', 'removed %']:.1f}% of ZIP codes. The removal leaves no trace in the "
    f"published file; the universe counts come from the release README."
))
findings.append((
    "F2",
    f"A second, visible absence sits inside the file: "
    f"{coverage.loc['zip', 'blank %']:.1f}% of published ZIP rows and "
    f"{coverage.loc['high_school', 'blank %']:.1f}% of high-school rows carry no estimate. "
    f"Counting both, usable coverage of the stated universe is "
    f"{coverage.loc['zip', 'usable % of universe']:.1f}% for ZIPs and "
    f"{coverage.loc['county', 'usable % of universe']:.1f}% for counties."
))
findings.append((
    "F3",
    f"The published standard error is documented as containing sampling error and privacy "
    f"noise together. Part of this dataset's uncertainty is therefore a policy parameter "
    f"(epsilon = 8), not a fact about the world."
))
slope_summary = ", ".join(
    f"{geography} {scaling.loc[geography, 'slope']:+.3f}" for geography in scaling.index
)
findings.append((
    "F4",
    f"The two components cannot be separated from the release. Regressing log standard error "
    f"on log unit size gives slopes of {slope_summary} -- none near "
    f"the -0.500 of sampling error or the -1.000 of Laplace noise scaling as 1/N -- because no "
    f"published column is the count the noise formula uses. The two county proxies correlate "
    f"{agreement[0]:.3f} in logs and give different slopes."
))
findings.append((
    "F5",
    f"County economic connectedness is the most precisely measured quantity in this corpus: "
    f"{n_units:,} units, median standard error {np.median(error):.4f}, 95% interval width "
    f"{2 * half_width / spread:.4f} of the between-county spread, reliability median "
    f"{np.median(reliability):.4f} with {(reliability >= RELIABILITY_FLOOR).mean():.1%} above "
    f"{RELIABILITY_FLOOR}."
))
findings.append((
    "F6",
    f"It still yields {int((span == 1).sum())} determined ranks out of {n_units:,}. The median "
    f"county's attainable rank span is {np.median(span):,.0f}, "
    f"{np.median(span) / n_units:.1%} of the table. Rank failure is structural: more units "
    f"makes it worse and more precision does not rescue it."
))
findings.append((
    "F7",
    f"Precision does buy the extremes. {int((span < 10).sum())} counties have an attainable "
    f"span under 10 and the tightest is {span.min()}, so the very top and very bottom of the "
    f"table are resolvable even though the order is not. "
    f"{int((span < 100).sum())} counties ({(span < 100).mean():.2%}) have a span under 100."
))
findings.append((
    "F8",
    f"Detection works where ranking does not: {(np.abs(z_score) > 1.96).mean():.1%} of counties "
    f"separate from the precision-weighted benchmark of {pooled:.4f} at 95%, and "
    f"{(np.abs(z_score) > 3.09).mean():.1%} at 99.8%."
))

for tag, text in findings:
    print(f"[{tag}] {text}")
    print()

[F1] The minimum-cell-size rule removes units before publication and the rate spans a factor of 92 across four geographies in one release: 0.2% of counties, 3.3% of colleges, 18.4% of high schools and 20.8% of ZIP codes. The removal leaves no trace in the published file; the universe counts come from the release README.

[F2] A second, visible absence sits inside the file: 17.6% of published ZIP rows and 19.7% of high-school rows carry no estimate. Counting both, usable coverage of the stated universe is 65.3% for ZIPs and 97.5% for counties.

[F3] The published standard error is documented as containing sampling error and privacy noise together. Part of this dataset's uncertainty is therefore a policy parameter (epsilon = 8), not a fact about the world.

[F4] The two components cannot be separated from the release. Regressing log standard error on log unit size gives slopes of county -0.308, zip -0.292, college -0.155, high_school -0.342 -- none near the -0.500 of sampling error or th

## Key Takeaways

1. **§7.2 is settled.** Tested at reliability 0.853 (D04), 0.916 (D10) and 0.995 here, on 133,
   133 and 3,018 units, the number of determined ranks is zero every time. Precision was never
   the binding constraint; multiplicity was.

2. **But precision buys the extremes.** The tails resolve even when the order does not. "Name
   the extremes, do not publish the list" is the operational form of this result, and it is
   more useful than a flat refusal to rank.

3. **A uniform suppression threshold is a patterned deletion.** The same rule removes 0.2% of
   counties and 20.8% of ZIP codes, because the rule responds to unit size and unit size is
   not randomly distributed over the things you want to conclude about.

4. **A file cannot report its own absences.** Units removed before publication leave no row,
   no flag and no gap. The universe count has to come from the documentation, and if nobody
   carries it into the code, the suppression rate silently becomes zero.

5. **Uncertainty can be partly a policy choice.** When a publisher combines statistical and
   disclosure error in one column, record that it did. Here the budget is published and the
   split still is not recoverable, because the noise depends on a network property that cannot
   be released without defeating the noise.

6. **Do not estimate the split from published size columns.** The regression in §2 is
   available, looks reasonable, and answers a different question than the one asked — the
   regressor is not the quantity the mechanism uses.

7. **A platform universe is not a population.** Facebook users aged 25–44 with at least 100 US
   friends, on one day. Coverage varies across places for reasons plausibly correlated with
   the social connectedness being measured, and no user count is published to check it with.

## Exercises

1. **Find the boundary of the extremes.** §3 shows a handful of counties with tiny rank spans.
   Determine how far down the table that holds — at what rank does the attainable span first
   exceed 50, and what does that imply about how long a published "top N" list could honestly
   be?

2. **Test the patterned-deletion claim.** The removed units are not in the file, so you cannot
   describe them directly. Using `pop2018` on the surviving ZIP codes and an external count of
   all ZIP codes, characterise what the missing fifth of the universe looks like — and state
   which part of your characterisation is measured and which is assumed.

3. **Re-run §3 on ZIP codes.** There are six times as many units and the standard errors are
   larger. Predict the number of determined ranks before running it, then run it. If your
   prediction was zero, say what result would have surprised you.

4. **Price the privacy budget.** Using only the published parameterisation, write down what
   would have to be released for the disclosure component to be separable, and explain why
   releasing it would defeat the mechanism. This is a proof, not a computation.

5. **Compare cohesiveness with connectedness.** `clustering_county` and `support_ratio_county`
   carry no standard errors at all. State precisely which of the claims made in this notebook
   about economic connectedness could still be made about clustering, and which could not.

## References

- Chetty, R., Jackson, M. O., Kuchler, T., Stroebel, J., et al. "Social capital I: measurement
  and associations with economic mobility." *Nature* 608, 108–121 (2022).
- Chetty, R., Jackson, M. O., Kuchler, T., Stroebel, J., et al. "Social capital II:
  determinants of economic connectedness." *Nature* 608, 122–134 (2022).
- Meta Platforms / Opportunity Insights. *Social Capital Atlas data release README*
  (31 July 2022) — the Laplace mechanism, ε = 8, the smoothed sensitivity term, the
  minimum-cell-size rule and its unit counts, and the definition of the standard-error columns.
- Meta Platforms. *Social Capital Atlas licence* — redistribution permitted with the licence
  attached; no combination with other sources that could identify an individual.
- Shen, W. and Louis, T. A. "Triple-goal estimates in two-stage hierarchical models."
  *Journal of the Royal Statistical Society B* 60(2), 455–471 (1998) — posterior means do not
  rank well, the result handbook §7.2 rests on.
- Handbook §7.1, §7.2 (ranking), §1.9 (Opportunity Atlas), §2 (firewall responses).

## Limitations

- **One snapshot, one release.** The network was observed on 28 May 2022 and there is no
  second vintage. Nothing here speaks to change over time, and no revision is possible to
  detect.
- **The scaling test is reported as a failure, not a result.** §2 establishes that the
  components cannot be separated from published columns. It does not establish the true
  scaling of either component, and the slopes should not be quoted as estimates of anything.
- **Reliability is computed from the published standard error**, which contains the privacy
  noise. So the reliability figure is itself partly a function of ε, and a different budget
  would give a different λ for the same network. §3's conclusion is unaffected — a *larger* λ
  would not create determined ranks — but the specific value is not a pure property of the
  measurement.
- **The rank bound is deliberately conservative.** `rank_intervals` uses exact interval
  overlap and assumes nothing about the joint distribution. A model-based interval would be
  narrower; it would also rest on assumptions this release does not license.
- **County-level only for the decisive test.** §3 runs on counties. The other three
  geographies are summarised in §4 but not put through the full rank analysis.
- **The universe cannot be audited.** No Facebook user count is published at any geography, so
  there is no way to check how platform coverage varies across places, and every statement
  here about representativeness rests on the publisher's own caveat rather than on
  measurement.